In [9]:
import pandas as pd
import numpy as np
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

data, meta = arff.loadarff("/Users/hari/Downloads/Bankruptcy/bankruptcy-prediction/data/raw/3year.arff")
df = pd.DataFrame(data)
df["class"] = df["class"].str.decode("utf-8").astype(int)

In [10]:
# Split first, before touching anything
X = df.drop(columns="class")
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Bankrupt share in train:", round(y_train.mean(), 4))
print("Bankrupt share in test: ", round(y_test.mean(), 4))

Train: (8402, 64)  Test: (2101, 64)
Bankrupt share in train: 0.0471
Bankrupt share in test:  0.0471


In [11]:
# Drop columns that are mostly empty (decided using TRAIN only)
missing_share = X_train.isna().mean()
too_empty = missing_share[missing_share > 0.40].index.tolist()
print("Dropping these columns:", too_empty)

X_train = X_train.drop(columns=too_empty)
X_test = X_test.drop(columns=too_empty)

Dropping these columns: ['Attr37']


In [12]:
# Fill the remaining blanks with the median
imputer = SimpleImputer(strategy="median")

X_train_clean = pd.DataFrame(
    imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_clean = pd.DataFrame(
    imputer.transform(X_test), columns=X_test.columns, index=X_test.index
)

print("Blanks left in train:", X_train_clean.isna().sum().sum())
print("Blanks left in test: ", X_test_clean.isna().sum().sum())

Blanks left in train: 0
Blanks left in test:  0


# Why the median? The median is the “middle” value, so crazy huge numbers don’t pull it around. The average would get dragged by outliers.

Notice: fit_transform on train means “learn the middle values from the training data and fill the blanks”. transform on test means “just use those same middle values”. The test data never teaches us anything.

In [14]:
# Save the clean files so we can use them in later steps
import os
os.makedirs("../data/processed", exist_ok=True)
X_train_clean.to_csv("../data/processed/X_train.csv", index=False)
X_test_clean.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)